# WTA Tennis Data Exploration

This notebook performs comprehensive exploratory data analysis on WTA tennis match data.

**Data Source:** https://github.com/ryantjx/tennis-match-data

**Dataset Overview:**
- 776,976+ records spanning 1967-2026
- Completed matches have `record_type='completed'`
- Fixtures have `record_type='fixture'`

In [1]:
# Import Required Libraries
import polars as pl
import altair as alt
import pandas as pd
import numpy as np
from datetime import datetime

# Enable Altair data transformer for larger datasets
alt.data_transformers.enable('default', max_rows=None)

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Load WTA Data

Load the WTA data from the parquet file using Polars.

In [2]:
# Load WTA data from parquet
url = "https://github.com/ryantjx/tennis-match-data/releases/download/data-latest/wta.parquet"

print(f"Loading data from: {url}")
df = pl.read_parquet(url)

print(f"\nDataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumn names:")
print(df.columns)

Loading data from: https://github.com/ryantjx/tennis-match-data/releases/download/data-latest/wta.parquet

Dataset loaded successfully!
Shape: (776926, 19)

Column names:
['date', 'match_id', 'tournament_id', 'tournament_name', 'tour', 'year', 'draw', 'round', 'format', 'player1_id', 'player1_name', 'player1_seed', 'player2_id', 'player2_name', 'player2_seed', 'winner_id', 'status', 'score', 'best_of']


In [3]:
# Display basic info about the dataset
print("Dataset Schema:")
print(df.schema)

print("\n\nFirst few rows:")
df.head()

Dataset Schema:
Schema({'date': Date, 'match_id': String, 'tournament_id': String, 'tournament_name': String, 'tour': String, 'year': Int16, 'draw': String, 'round': String, 'format': String, 'player1_id': List(String), 'player1_name': List(String), 'player1_seed': String, 'player2_id': List(String), 'player2_name': List(String), 'player2_seed': String, 'winner_id': List(String), 'status': String, 'score': String, 'best_of': Int8})


First few rows:


date,match_id,tournament_id,tournament_name,tour,year,draw,round,format,player1_id,player1_name,player1_seed,player2_id,player2_name,player2_seed,winner_id,status,score,best_of
date,str,str,str,str,i16,str,str,str,list[str],list[str],str,list[str],list[str],str,list[str],str,str,i8
null,"""match:wta:992a2784d983b9cde637""","""tournament_wta_1968_00e8c9f20d…","""Locust Valley""","""wta""",1968,"""main""","""F""","""singles""","[""wta:200175""]","[""Mary Ann Eisel""]",null,"[""wta:200168""]","[""Karen Krantzcke""]",null,"[""wta:200175""]","""completed""","""3-6 15-13 8-6""",3
null,"""match:wta:318dc528b9528bfc4261""","""tournament_wta_1968_00e8c9f20d…","""Locust Valley""","""wta""",1968,"""main""","""R16""","""singles""","[""wta:200175""]","[""Mary Ann Eisel""]",null,"[""wta:201103""]","[""Pixie Lamm""]",null,"[""wta:200175""]","""completed""","""6-3 6-3""",3
null,"""match:wta:399e07339946a287fb55""","""tournament_wta_1968_00e8c9f20d…","""Locust Valley""","""wta""",1968,"""main""","""R16""","""singles""","[""wta:200871""]","[""Tory Ann Fretz""]",null,"[""wta:200779""]","[""Maryna Godwin""]",null,"[""wta:200871""]","""completed""","""6-2 9-7""",3
null,"""match:wta:406216dc2a3271bd36a5""","""tournament_wta_1968_00e8c9f20d…","""Locust Valley""","""wta""",1968,"""main""","""R16""","""singles""","[""wta:201115""]","[""Victoria Rogers""]",null,"[""wta:200286""]","[""Peggy Michel""]",null,"[""wta:201115""]","""completed""","""6-3 6-1""",3
null,"""match:wta:5796347fd8e53ac55c2e""","""tournament_wta_1968_00e8c9f20d…","""Locust Valley""","""wta""",1968,"""main""","""R16""","""singles""","[""wta:200168""]","[""Karen Krantzcke""]",null,"[""wta:201107""]","[""Nadine Netter""]",null,"[""wta:200168""]","""completed""","""6-2 6-1""",3


In [5]:
df.filter(
    pl.col('date').is_not_null()
)

date,match_id,tournament_id,tournament_name,tour,year,draw,round,format,player1_id,player1_name,player1_seed,player2_id,player2_name,player2_seed,winner_id,status,score,best_of
date,str,str,str,str,i16,str,str,str,list[str],list[str],str,list[str],list[str],str,list[str],str,str,i8


## 2. Data Preprocessing

Process dates correctly using `event_start_date` to extract year (the `year` column has null values).

In [ ]:
# Process dates - event_start_date is already a date type
df_processed = df.with_columns(
    pl.col("event_start_date").alias("match_date")
).with_columns(
    pl.col("match_date").dt.year().alias("year"),
    pl.col("match_date").dt.month().alias("month"),
    (pl.col("match_date").dt.year() // 10 * 10).alias("decade")
)

# Filter out null dates
df_processed = df_processed.filter(pl.col("match_date").is_not_null())

print(f"Processed dataset shape: {df_processed.shape}")
print(f"\nYear range: {df_processed['year'].min()} - {df_processed['year'].max()}")
print(f"\nRecord types:")
print(df_processed['record_type'].value_counts())

SchemaError: invalid series dtype: expected `String`, got `date` for series with name `event_start_date`

This error occurred in the following expression:
	col("event_start_date").str.strptime(["raise"])


## 3. Comprehensive Summary Dashboard

In [ ]:
# Create comprehensive summary
total_records = df_processed.height
unique_players = df_processed.select(pl.col("player_1_name"), pl.col("player_2_name")).melt().select(pl.col("value").n_unique()).item()
unique_tournaments = df_processed['event_name'].n_unique()
year_min = df_processed['year'].min()
year_max = df_processed['year'].max()

# Surface distribution
surface_dist = df_processed['surface'].value_counts(normalize=True).with_columns(
    (pl.col("proportion") * 100).round(2).alias("percentage")
)

# Tournament level breakdown
level_dist = df_processed['tournament_level'].value_counts().sort("count", descending=True)

# Top 5 players by match count
player_matches = df_processed.select(pl.col("player_1_name"), pl.col("player_2_name")).melt().group_by("value").count().sort("count", descending=True).head(5)

print("=
60
,
WTA TENNIS DATA SUMMARY DASHBOARD")
print("=
60
,
\n📊 Total Records: {total_records:,}")
print(f"👥 Unique Players: {unique_players:,}")
print(f"🏆 Unique Tournaments: {unique_tournaments:,}")
print(f"📅 Year Range: {year_min} - {year_max}")

print("\n📈 Surface Distribution:")
for row in surface_dist.iter_rows(named=True):
    print(f"  {row['surface']}: {row['percentage']:.1f}% ({row['count']:,} matches)")

print("\n🏅 Tournament Level Breakdown:")
for row in level_dist.iter_rows(named=True):
    print(f"  {row['tournament_level']}: {row['count']:,} matches")

print("\n⭐ Top 5 Players by Match Count:")
for i, row in enumerate(player_matches.iter_rows(named=True), 1):
    print(f"  {i}. {row['value']}: {row['count']:,} matches")
print("\n" + "=" * 60)

## 4. Match Distribution by Year (with Rolling Average)

In [ ]:
# Match distribution by year with rolling average
matches_by_year = df_processed.group_by("year").agg(
    pl.count().alias("match_count")
).sort("year")

# Calculate 3-year rolling average
matches_by_year_pd = matches_by_year.to_pandas()
matches_by_year_pd["rolling_avg"] = matches_by_year_pd["match_count"].rolling(window=3, center=True, min_periods=1).mean()

# Create the chart
base = alt.Chart(matches_by_year_pd).encode(
    x=alt.X("year:O", title="Year", axis=alt.Axis(labelAngle=-45)),
    tooltip=["year", "match_count", "rolling_avg"]
)

bars = base.mark_bar(color="steelblue", opacity=0.7).encode(
    y=alt.Y("match_count:Q", title="Number of Matches")
)

line = base.mark_line(color="red", strokeWidth=2).encode(
    y=alt.Y("rolling_avg:Q", title="3-Year Rolling Average")
)

chart = (bars + line).properties(
    title="WTA Matches by Year with 3-Year Rolling Average",
    width=800,
    height=400
).resolve_scale(
    y="independent"
)

chart.show()

## 5. Surface Type Distribution (Pie Chart)

In [ ]:
# Surface type distribution pie chart
surface_counts = df_processed['surface'].value_counts().sort("count", descending=True)

# Define colors for surfaces
surface_colors = {
    "Hard": "#4472C4",
    "Clay": "#E7A428",
    "Grass": "#70AD47",
    "Carpet": "#C55A11",
    "Unknown": "#7F7F7F"
}

pie_chart = alt.Chart(surface_counts.to_pandas()).mark_arc().encode(
    theta=alt.Theta("count:Q", stack=True),
    color=alt.Color("surface:N", scale=alt.Scale(domain=list(surface_colors.keys()), range=list(surface_colors.values()))),
    tooltip=["surface", "count", alt.Tooltip("count:Q", format=".0%", title="Percentage", stack="normalize")]
).properties(
    title="Surface Type Distribution",
    width=400,
    height=400
)

pie_chart.show()

## 6. Tournament Level Distribution

In [ ]:
# Tournament level distribution
level_counts = df_processed['tournament_level'].value_counts().sort("count", descending=True)

level_chart = alt.Chart(level_counts.to_pandas()).mark_bar().encode(
    x=alt.X("count:Q", title="Number of Matches"),
    y=alt.Y("tournament_level:N", title="Tournament Level", sort="-x"),
    color=alt.Color("tournament_level:N", legend=None),
    tooltip=["tournament_level", "count"]
).properties(
    title="Tournament Level Distribution",
    width=600,
    height=300
)

level_chart.show()

## 7. Top 15 Players by Match Count

In [ ]:
# Top 15 players by match count
player_matches = df_processed.select(
    pl.col("player_1_name"),
    pl.col("player_2_name")
).melt().group_by("value").count().sort("count", descending=True).head(15)

players_chart = alt.Chart(player_matches.to_pandas()).mark_bar().encode(
    x=alt.X("count:Q", title="Number of Matches"),
    y=alt.Y("value:N", title="Player", sort="-x"),
    color=alt.Color("count:Q", scale=alt.Scale(scheme="blues"), legend=None),
    tooltip=["value", "count"]
).properties(
    title="Top 15 Players by Match Count",
    width=600,
    height=400
)

players_chart.show()

## 8. Top 15 Tournaments by Match Count

In [ ]:
# Top 15 tournaments by match count
# Calculate years held using match_date for year calculations
tournament_stats = df_processed.group_by("event_name").agg(
    pl.count().alias("match_count"),
    pl.col("match_date").dt.year().min().alias("first_year"),
    pl.col("match_date").dt.year().max().alias("last_year")
).with_columns(
    (pl.col("last_year") - pl.col("first_year") + 1).alias("years_held")
).sort("match_count", descending=True).head(15)

tournaments_chart = alt.Chart(tournament_stats.to_pandas()).mark_bar().encode(
    x=alt.X("match_count:Q", title="Number of Matches"),
    y=alt.Y("event_name:N", title="Tournament", sort="-x"),
    color=alt.Color("match_count:Q", scale=alt.Scale(scheme="greens"), legend=None),
    tooltip=["event_name", "match_count", "first_year", "last_year", "years_held"]
).properties(
    title="Top 15 Tournaments by Match Count",
    width=600,
    height=400
)

tournaments_chart.show()

## 9. Surface Evolution by Decade (Stacked Area Chart)

In [ ]:
# Surface evolution by decade
surface_by_decade = df_processed.filter(
    pl.col("surface").is_not_null()
).group_by(["decade", "surface"]).agg(
    pl.count().alias("match_count")
).sort(["decade", "surface"])

# Create stacked area chart
surface_evolution = alt.Chart(surface_by_decade.to_pandas()).mark_area().encode(
    x=alt.X("decade:O", title="Decade"),
    y=alt.Y("match_count:Q", title="Number of Matches", stack="normalize"),
    color=alt.Color("surface:N", scale=alt.Scale(scheme="category10")),
    tooltip=["decade", "surface", "match_count"]
).properties(
    title="Surface Evolution by Decade (Proportional)",
    width=700,
    height=400
)

surface_evolution.show()

## 10. Player Win Rate Distribution

In [ ]:
# Calculate player win rates
# Get wins for each player
player_wins = df_processed.filter(
    pl.col("record_type") == "completed"
).select(
    pl.col("player_1_name").alias("player"),
    pl.when(pl.col("player_1_score") > pl.col("player_2_score")).then(1).otherwise(0).alias("win")
).vstack(
    df_processed.filter(
        pl.col("record_type") == "completed"
    ).select(
        pl.col("player_2_name").alias("player"),
        pl.when(pl.col("player_2_score") > pl.col("player_1_score")).then(1).otherwise(0).alias("win")
    )
)

# Calculate win rates for players with at least 50 matches
player_stats = player_wins.group_by("player").agg(
    pl.count().alias("total_matches"),
    pl.sum("win").alias("wins")
).with_columns(
    (pl.col("wins") / pl.col("total_matches") * 100).alias("win_rate")
).filter(
    pl.col("total_matches") >= 50
).sort("win_rate", descending=True)

print(f"Players with 50+ matches: {player_stats.height}")

# Create histogram of win rates
win_rate_hist = alt.Chart(player_stats.to_pandas()).mark_bar().encode(
    x=alt.X("win_rate:Q", bin=alt.Bin(step=5), title="Win Rate (%)"),
    y=alt.Y("count()", title="Number of Players"),
    tooltip=[alt.Tooltip("win_rate:Q", bin=True, title="Win Rate Range"), "count()"]
).properties(
    title="Player Win Rate Distribution (Players with 50+ Matches)",
    width=600,
    height=400
)

win_rate_hist.show()

## 11. Surface vs Tournament Level Heatmap

In [ ]:
# Surface vs Tournament Level Heatmap
surface_level = df_processed.filter(
    pl.col("surface").is_not_null(),
    pl.col("tournament_level").is_not_null()
).group_by(["surface", "tournament_level"]).agg(
    pl.count().alias("match_count")
)

heatmap = alt.Chart(surface_level.to_pandas()).mark_rect().encode(
    x=alt.X("tournament_level:N", title="Tournament Level"),
    y=alt.Y("surface:N", title="Surface"),
    color=alt.Color("match_count:Q", scale=alt.Scale(scheme="yelloworangered"), title="Matches"),
    tooltip=["surface", "tournament_level", "match_count"]
).properties(
    title="Surface vs Tournament Level Heatmap",
    width=500,
    height=300
)

heatmap.show()

## 12. Tournament Level vs Surface Bubble Chart

In [ ]:
# Tournament Level vs Surface Bubble Chart
bubble_data = df_processed.filter(
    pl.col("surface").is_not_null(),
    pl.col("tournament_level").is_not_null()
).group_by(["surface", "tournament_level"]).agg(
    pl.count().alias("match_count"),
    pl.col("match_date").dt.year().mean().alias("avg_year")
)

bubble_chart = alt.Chart(bubble_data.to_pandas()).mark_circle().encode(
    x=alt.X("tournament_level:N", title="Tournament Level"),
    y=alt.Y("surface:N", title="Surface"),
    size=alt.Size("match_count:Q", scale=alt.Scale(range=[100, 2000]), title="Match Count"),
    color=alt.Color("avg_year:Q", scale=alt.Scale(scheme="viridis"), title="Avg Year"),
    tooltip=["surface", "tournament_level", "match_count", alt.Tooltip("avg_year:Q", format=".0f")]
).properties(
    title="Tournament Level vs Surface (Bubble Size = Match Count)",
    width=500,
    height=300
)

bubble_chart.show()

## 13. Future Fixtures Summary

In [ ]:
# Future fixtures summary
fixtures = df_processed.filter(pl.col("record_type") == "fixture")

if fixtures.height > 0:
    print(f"\n📅 Future Fixtures Summary")
    print(f"Total Fixtures: {fixtures.height:,}")
    
    # Fixtures by year
    fixtures_by_year = fixtures.group_by("year").agg(
        pl.count().alias("fixture_count")
    ).sort("year")
    
    print("\nFixtures by Year:")
    for row in fixtures_by_year.iter_rows(named=True):
        print(f"  {row['year']}: {row['fixture_count']:,} fixtures")
    
    # Fixtures by surface",
count", descending=True)
    
    print("\nFixtures by Surface:")
    for row in fixtures_by_surface.iter_rows(named=True):
        print(f"  {row['surface']}: {row['count']:,} fixtures")
    
    # Fixtures by tournament level
    fixtures_by_level = fixtures['tournament_level'].value_counts().sort("count", descending=True)
    
    print("\nFixtures by Tournament Level:")
    for row in fixtures_by_level.iter_rows(named=True):
        print(f"  {row['tournament_level']}: {row['count']:,} fixtures")
    
    # Visualize fixtures by year
    fixtures_chart = alt.Chart(fixtures_by_year.to_pandas()).mark_bar(color="coral").encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("fixture_count:Q", title="Number of Fixtures"),
        tooltip=["year", "fixture_count"]
    ).properties(
        title="Future Fixtures by Year",
        width=600,
        height=300
    )
    
    fixtures_chart.show()
else:
    print("\n📅 No future fixtures found in the dataset.")

## 14. Additional Insights

In [ ]:
# Additional insights
print("\n" + "=" * 60)
print("ADDITIONAL INSIGHTS")
print("=" * 60)

# Completed vs Fixtures ratio
completed_count = df_processed.filter(pl.col("record_type") == "completed").height
fixture_count = df_processed.filter(pl.col("record_type") == "fixture").height

print(f"\n📊 Record Type Distribution:")
print(f"  Completed Matches: {completed_count:,} ({completed_count/total_records*100:.1f}%)")
print(f"  Fixtures: {fixture_count:,} ({fixture_count/total_records*100:.1f}%)")

# Most active year
most_active_year = matches_by_year.sort("count", descending=True).head(1)
for row in most_active_year.iter_rows(named=True):
    print(f"\n🏆 Most Active Year: {row['year']} with {row['count']:,} matches")

# Average matches per year
avg_matches = matches_by_year['count'].mean()
print(f"📈 Average Matches per Year: {avg_matches:,.0f}")

# Surface diversity over time
surface_diversity = df_processed.filter(
    pl.col("surface").is_not_null()
).group_by("year").agg(
    pl.col("surface").n_unique().alias("unique_surfaces")
).sort("year")

avg_surfaces = surface_diversity['unique_surfaces'].mean()
print(f"🎾 Average Surface Types per Year: {avg_surfaces:.1f}")

print("\n" + "=" * 60)

## Conclusion

This notebook has provided a comprehensive exploration of the WTA tennis match data, including:

1. **Data Overview**: Loaded and processed 776,976+ records spanning 1967-2026
2. **Temporal Analysis**: Match distribution by year with rolling averages
3. **Surface Analysis**: Distribution and evolution across decades
4. **Player Analysis**: Top players by match count and win rate distributions
5. **Tournament Analysis**: Top tournaments and level distributions
6. **Correlation Analysis**: Surface vs tournament level relationships
7. **Future Fixtures**: Summary of upcoming matches

The visualizations reveal interesting patterns in tennis data, such as the dominance of hard court surfaces, the growth of the sport over decades, and the distribution of matches across different tournament levels.

# WTA Tennis Data Exploration

This notebook performs exploratory data analysis on WTA (Women's Tennis Association) match data from the tennis-match-data repository.

## Overview
- Load WTA match data from Parquet files
- Explore match distributions, player statistics, and trends
- Visualize patterns in the data

## 1. Import Required Libraries

Import polars for fast data manipulation, altair for visualization, and other supporting libraries.

In [56]:
# Import required libraries
import polars as pl
import altair as alt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configure Altair for better rendering
alt.data_transformers.disable_max_rows()

print("Libraries imported successfully!")
print(f"Polars version: {pl.__version__}")

Libraries imported successfully!
Polars version: 1.42.1


## 2. Load WTA Dataset

Load WTA match data from the tennis-match-data repository's rolling release.

In [57]:
# URL for WTA data from tennis-match-data repository
WTA_URL = "https://github.com/ryantjx/tennis-match-data/releases/download/data-latest/wta.parquet"

print("Loading WTA match data...")
print(f"Source: {WTA_URL}")

# Load the Parquet file
df = pl.read_parquet(WTA_URL)

print(f"\nDataset loaded successfully!")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loading WTA match data...
Source: https://github.com/ryantjx/tennis-match-data/releases/download/data-latest/wta.parquet

Dataset loaded successfully!
Shape: 776,976 rows × 55 columns


## 3. Data Overview and Schema

Examine the dataset structure, column types, and sample records.

In [58]:
# Display column information
print("Column Names and Data Types:")
print("-" * 50)
for col, dtype in zip(df.columns, df.dtypes):
    print(f"{col:<30} {dtype}")

print(f"\nTotal columns: {len(df.columns)}")
print(f"Total rows: {df.shape[0]:,}")

Column Names and Data Types:
--------------------------------------------------
record_type                    String
record_id                      String
is_fixture                     Boolean
fixture_id                     String
scheduled_on                   Date
scheduled_at                   Datetime(time_unit='us', time_zone=None)
schedule_date_source           String
fixture_observed_on            Date
fixture_source                 String
fixture_source_match_id        String
match_id                       String
event_id                       String
tour                           String
year                           Int64
discipline                     String
event_name                     String
level                          String
level_detail                   String
source_level                   String
surface                        String
indoor                         Boolean
event_start_date               Date
event_city                     String
event_country    

In [59]:
# Display first few rows
print("Sample Data (first 5 rows):")
df.head()

Sample Data (first 5 rows):


record_type,record_id,is_fixture,fixture_id,scheduled_on,scheduled_at,schedule_date_source,fixture_observed_on,fixture_source,fixture_source_match_id,match_id,event_id,tour,year,discipline,event_name,level,level_detail,source_level,surface,indoor,event_start_date,event_city,event_country,draw,round,round_order,bracket_slot,played_on,played_on_precision,player1_id,player1_name,player1_country,player2_id,player2_name,player2_country,winner_id,loser_id,winner_side,player1_seed,player2_seed,player1_entry,player2_entry,player1_rank,player2_rank,player1_rank_points,player2_rank_points,status,termination,score,best_of,first_observed_on,last_updated_on,preferred_source,source_count
str,str,bool,str,date,datetime[μs],str,date,str,str,str,str,str,i64,str,str,str,str,str,str,bool,date,str,str,str,str,i16,str,date,str,str,str,str,str,str,str,str,str,i8,str,str,str,str,i32,i32,i32,i32,str,str,str,i8,date,date,str,i16
"""completed""","""match:wta:339f8825650850ee358b""",false,null,null,null,null,null,null,null,"""match:wta:339f8825650850ee358b""","""event:wta:09379a924b31b435b080""","""wta""",1968,"""singles""","""Port Elizabeth""","""other""","""W""","""W""","""hard""",false,1967-12-25,null,null,"""main""","""R32""",30,null,null,"""event_only""","""wta:200211""","""Winnie Shaw""","""GBR""","""wta:223727""","""Lynn Mcdonald""","""RSA""","""wta:200211""","""wta:223727""",1,null,null,null,null,null,null,null,null,"""completed""",null,"""6-2 3-6 6-1""",3,2026-07-12,2026-07-12,"""sackmann""",1
"""completed""","""match:wta:5eb5448f75bb67887dc7""",false,null,null,null,null,null,null,null,"""match:wta:5eb5448f75bb67887dc7""","""event:wta:09379a924b31b435b080""","""wta""",1968,"""singles""","""Port Elizabeth""","""other""","""W""","""W""","""hard""",false,1967-12-25,null,null,"""main""","""R32""",30,null,null,"""event_only""","""wta:200901""","""Nell Truman""","""GBR""","""wta:200891""","""Daphne Pattison""","""ZIM""","""wta:200901""","""wta:200891""",1,null,null,null,null,null,null,null,null,"""completed""",null,"""8-6 6-0""",3,2026-07-12,2026-07-12,"""sackmann""",1
"""completed""","""match:wta:08a0b035de340ba71837""",false,null,null,null,null,null,null,null,"""match:wta:08a0b035de340ba71837""","""event:wta:09379a924b31b435b080""","""wta""",1968,"""singles""","""Port Elizabeth""","""other""","""W""","""W""","""hard""",false,1967-12-25,null,null,"""main""","""R16""",40,null,null,"""event_only""","""wta:200901""","""Nell Truman""","""GBR""","""wta:200826""","""Greta Delport""","""RSA""","""wta:200901""","""wta:200826""",1,null,null,null,null,null,null,null,null,"""completed""",null,"""6-2 7-5""",3,2026-07-12,2026-07-12,"""sackmann""",1
"""completed""","""match:wta:0a685ee5b69f1e1650bc""",false,null,null,null,null,null,null,null,"""match:wta:0a685ee5b69f1e1650bc""","""event:wta:09379a924b31b435b080""","""wta""",1968,"""singles""","""Port Elizabeth""","""other""","""W""","""W""","""hard""",false,1967-12-25,null,null,"""main""","""R16""",40,null,null,"""event_only""","""wta:200808""","""Laura Rossouw""","""RSA""","""wta:200831""","""Carmen Coronado""","""ESP""","""wta:200808""","""wta:200831""",1,null,null,null,null,null,null,null,null,"""completed""",null,"""10-8 6-2""",3,2026-07-12,2026-07-12,"""sackmann""",1
"""completed""","""match:wta:1855f0aaf59b357b8a31""",false,null,null,null,null,null,null,null,"""match:wta:1855f0aaf59b357b8a31""","""event:wta:09379a924b31b435b080""","""wta""",1968,"""singles""","""Port Elizabeth""","""other""","""W""","""W""","""hard""",false,1967-12-25,null,null,"""main""","""R16""",40,null,null,"""event_only""","""wta:201126""","""Carole Graebner""","""USA""","""wta:201122""","""Anita Van Deventer""","""RSA""","""wta:201126""","""wta:201122""",1,null,null,null,null,null,null,null,null,"""completed""",null,"""6-2 6-1""",3,2026-07-12,2026-07-12,"""sackmann""",1


## 4. Data Cleaning and Preprocessing

Handle missing values, convert date columns, and prepare data for analysis.

In [60]:
# Check for null values
print("Null Value Counts:")
print("-" * 40)
null_counts = df.null_count()
for col in df.columns:
    count = null_counts[col][0]
    pct = (count / df.shape[0]) * 100
    if count > 0:
        print(f"{col:<30} {count:>8,} ({pct:.1f}%)")

# Check if we have a date column and its format
print("\n\nDate column analysis:")
if "event_start_date" in df.columns:
    print("Using 'event_start_date' column for dates")
    date_col = "event_start_date"
elif "played_on" in df.columns:
    print("Using 'played_on' column for dates")
    date_col = "played_on"
else:
    date_col = None
    print("Available date-related columns:", [c for c in df.columns if "date" in c.lower()])

if date_col:
    print(f"Date column type: {df[date_col].dtype}")
    non_null_dates = df[date_col].drop_nulls()
    if non_null_dates.shape[0] > 0:
        print(f"Date range: {non_null_dates.min()} to {non_null_dates.max()}")
        print(f"Non-null date count: {non_null_dates.shape[0]:,}")
    else:
        print("Warning: Date column has no non-null values")
        print(f"Sample values: {df[date_col].head(5).to_list()}")

Null Value Counts:
----------------------------------------
fixture_id                      776,962 (100.0%)
scheduled_on                    776,976 (100.0%)
scheduled_at                    776,976 (100.0%)
schedule_date_source            776,962 (100.0%)
fixture_observed_on             776,962 (100.0%)
fixture_source                  776,962 (100.0%)
fixture_source_match_id         776,962 (100.0%)
match_id                             14 (0.0%)
level_detail                      2,248 (0.3%)
source_level                      2,248 (0.3%)
surface                           8,441 (1.1%)
indoor                               14 (0.0%)
event_start_date                  2,248 (0.3%)
event_city                      776,976 (100.0%)
event_country                   776,976 (100.0%)
round_order                          14 (0.0%)
bracket_slot                    774,728 (99.7%)
played_on                       776,976 (100.0%)
played_on_precision                  14 (0.0%)
player1_id                

In [61]:
# Convert date column and extract year for analysis
if date_col:
    df = df.with_columns(
        pl.col(date_col).cast(pl.Date).alias("match_date"),
        pl.col(date_col).cast(pl.Date).dt.year().alias("year")
    )
    
    # Check if year extraction worked
    year_min = df['year'].drop_nulls().min()
    year_max = df['year'].drop_nulls().max()
    
    if year_min is not None and year_max is not None:
        print("Date conversion successful!")
        print(f"Year range: {year_min} - {year_max}")
        print(f"Records with valid dates: {df['match_date'].drop_nulls().shape[0]:,}")
    else:
        print("Warning: Year extraction resulted in null values")
else:
    print("No date column found for conversion")

Date conversion successful!
Year range: 1967 - 2026
Records with valid dates: 774,728


## 5. Exploratory Data Analysis - Summary Statistics

Generate descriptive statistics and examine data distributions.

In [62]:
# Basic summary statistics
print("Dataset Summary:")
print("=" * 60)

# Count unique values for key columns
if "player1_name" in df.columns and "player2_name" in df.columns:
    unique_players = len(set(df["player1_name"].drop_nulls().to_list() + 
                            df["player2_name"].drop_nulls().to_list()))
    print(f"Unique players: {unique_players:,}")

if "event_name" in df.columns:
    print(f"Unique tournaments: {df['event_name'].n_unique():,}")

if "surface" in df.columns:
    print(f"Surfaces: {df['surface'].n_unique()} types")
    print(f"  - {df['surface'].drop_nulls().unique().to_list()}")

if "level" in df.columns:
    print(f"Tournament levels: {df['level'].n_unique()} types")

if "year" in df.columns:
    print(f"Years covered: {df['year'].min()} - {df['year'].max()}")
    print(f"Total years: {df['year'].n_unique()}")

print(f"\nTotal matches: {df.shape[0]:,}")

Dataset Summary:
Unique players: 30,365
Unique tournaments: 11,079
Surfaces: 5 types
  - ['clay', 'hard', 'grass', 'carpet']
Tournament levels: 10 types
Years covered: 1967 - 2026
Total years: 61

Total matches: 776,976


## 6. Visualize Match Distribution by Year

Create a time series visualization of match counts over the years.

In [63]:
# Matches per year
if "year" in df.columns:
    matches_by_year = df.group_by("year").agg(
        pl.count().alias("match_count")
    ).sort("year")
    
    print("Matches by Year:")
    print(matches_by_year)
    
    # Create visualization
    chart = alt.Chart(matches_by_year.to_pandas()).mark_bar(color="steelblue").encode(
        x=alt.X("year:O", title="Year", axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("match_count:Q", title="Number of Matches"),
        tooltip=["year", "match_count"]
    ).properties(
        title="WTA Matches per Year",
        width=700,
        height=400
    )
    
    chart.show()

Matches by Year:
shape: (61, 2)
┌──────┬─────────────┐
│ year ┆ match_count │
│ ---  ┆ ---         │
│ i32  ┆ u32         │
╞══════╪═════════════╡
│ null ┆ 2248        │
│ 1967 ┆ 21          │
│ 1968 ┆ 3180        │
│ 1969 ┆ 3218        │
│ 1970 ┆ 3214        │
│ …    ┆ …           │
│ 2022 ┆ 35598       │
│ 2023 ┆ 37028       │
│ 2024 ┆ 41057       │
│ 2025 ┆ 23818       │
│ 2026 ┆ 10068       │
└──────┴─────────────┘


alt.Chart(...)

## 7. Surface Type Analysis

Analyze match distribution and patterns by court surface.

In [64]:
# Surface distribution
if "surface" in df.columns:
    surface_counts = df.group_by("surface").agg(
        pl.count().alias("match_count")
    ).sort("match_count", descending=True)
    
    print("Matches by Surface:")
    print(surface_counts)
    
    # Create pie chart
    chart = alt.Chart(surface_counts.to_pandas()).mark_arc(innerRadius=50).encode(
        theta=alt.Theta("match_count:Q", stack=True),
        color=alt.Color("surface:N", title="Surface"),
        tooltip=["surface", "match_count"]
    ).properties(
        title="Match Distribution by Surface",
        width=400,
        height=400
    )
    
    chart.show()

Matches by Surface:
shape: (5, 2)
┌─────────┬─────────────┐
│ surface ┆ match_count │
│ ---     ┆ ---         │
│ str     ┆ u32         │
╞═════════╪═════════════╡
│ hard    ┆ 358835      │
│ clay    ┆ 328429      │
│ carpet  ┆ 41462       │
│ grass   ┆ 39809       │
│ null    ┆ 8441        │
└─────────┴─────────────┘


alt.Chart(...)

## 8. Tournament Level Analysis

Examine the distribution of matches across different tournament levels.

In [65]:
# Tournament level distribution
if "level" in df.columns:
    level_counts = df.group_by("level").agg(
        pl.count().alias("match_count")
    ).sort("match_count", descending=True)
    
    print("Matches by Tournament Level:")
    print(level_counts)
    
    # Create bar chart
    chart = alt.Chart(level_counts.to_pandas()).mark_bar().encode(
        x=alt.X("level:N", title="Tournament Level", sort="-y"),
        y=alt.Y("match_count:Q", title="Number of Matches"),
        color=alt.Color("level:N", legend=None),
        tooltip=["level", "match_count"]
    ).properties(
        title="Match Distribution by Tournament Level",
        width=600,
        height=400
    )
    
    chart.show()

Matches by Tournament Level:
shape: (10, 2)
┌─────────────┬─────────────┐
│ level       ┆ match_count │
│ ---         ┆ ---         │
│ str         ┆ u32         │
╞═════════════╪═════════════╡
│ itf         ┆ 472950      │
│ other       ┆ 124905      │
│ wta_125     ┆ 67955       │
│ grand_slam  ┆ 41369       │
│ wta_500     ┆ 26270       │
│ wta_1000    ┆ 15055       │
│ wta_250     ┆ 15021       │
│ team        ┆ 12246       │
│ olympics    ┆ 672         │
│ tour_finals ┆ 533         │
└─────────────┴─────────────┘


alt.Chart(...)

## 9. Top Players Analysis

Identify the most active players by match count and analyze their win rates.

In [66]:
# Top players by match count
if "player1_name" in df.columns and "player2_name" in df.columns and "winner" in df.columns:
    # Count matches per player
    player1_counts = df.group_by("player1_name").agg(pl.count().alias("matches_as_p1"))
    player2_counts = df.group_by("player2_name").agg(pl.count().alias("matches_as_p2"))
    
    # Combine counts
    all_players = pl.concat([
        df.select(pl.col("player1_name").alias("player")),
        df.select(pl.col("player2_name").alias("player"))
    ]).group_by("player").agg(pl.count().alias("total_matches")).sort("total_matches", descending=True)
    
    print("Top 20 Players by Match Count:")
    print(all_players.head(20))
    
    # Visualize top 15
    top_15 = all_players.head(15)
    
    chart = alt.Chart(top_15.to_pandas()).mark_bar(color="coral").encode(
        x=alt.X("total_matches:Q", title="Total Matches"),
        y=alt.Y("player:N", title="Player", sort="-x"),
        tooltip=["player", "total_matches"]
    ).properties(
        title="Top 15 Players by Match Count",
        width=600,
        height=500
    )
    
    chart.show()

## 10. Round Distribution Analysis

Examine how matches are distributed across tournament rounds.

In [67]:
# Round distribution
if "round" in df.columns:
    round_counts = df.group_by("round").agg(
        pl.count().alias("match_count")
    ).sort("match_count", descending=True)
    
    print("Matches by Round:")
    print(round_counts)
    
    # Create horizontal bar chart
    chart = alt.Chart(round_counts.to_pandas()).mark_bar(color="seagreen").encode(
        x=alt.X("match_count:Q", title="Number of Matches"),
        y=alt.Y("round:N", title="Round", sort="-x"),
        tooltip=["round", "match_count"]
    ).properties(
        title="Match Distribution by Tournament Round",
        width=600,
        height=400
    )
    
    chart.show()

Matches by Round:
shape: (15, 2)
┌────────────────────────┬─────────────┐
│ round                  ┆ match_count │
│ ---                    ┆ ---         │
│ str                    ┆ u32         │
╞════════════════════════╪═════════════╡
│ R32                    ┆ 284932      │
│ R16                    ┆ 148555      │
│ Q1                     ┆ 85531       │
│ QF                     ┆ 75652       │
│ Q2                     ┆ 52412       │
│ …                      ┆ …           │
│ RR                     ┆ 13078       │
│ Q4                     ┆ 259         │
│ Qualifying competition ┆ 226         │
│ BR                     ┆ 220         │
│ Q5                     ┆ 20          │
└────────────────────────┴─────────────┘


alt.Chart(...)

## 11. Surface vs Tournament Level Cross-Analysis

Analyze the relationship between surface types and tournament levels.

In [68]:
# Cross-tabulation of surface and level
if "surface" in df.columns and "level" in df.columns:
    cross_tab = df.group_by(["surface", "level"]).agg(
        pl.count().alias("match_count")
    ).sort("match_count", descending=True)
    
    print("Surface vs Tournament Level:")
    print(cross_tab.head(20))
    
    # Create heatmap
    chart = alt.Chart(cross_tab.to_pandas()).mark_rect().encode(
        x=alt.X("level:N", title="Tournament Level"),
        y=alt.Y("surface:N", title="Surface"),
        color=alt.Color("match_count:Q", title="Matches", scale=alt.Scale(scheme="blues")),
        tooltip=["surface", "level", "match_count"]
    ).properties(
        title="Match Count: Surface vs Tournament Level",
        width=600,
        height=300
    )
    
    chart.show()

Surface vs Tournament Level:
shape: (20, 3)
┌─────────┬─────────┬─────────────┐
│ surface ┆ level   ┆ match_count │
│ ---     ┆ ---     ┆ ---         │
│ str     ┆ str     ┆ u32         │
╞═════════╪═════════╪═════════════╡
│ hard    ┆ itf     ┆ 226526      │
│ clay    ┆ itf     ┆ 223830      │
│ hard    ┆ other   ┆ 45252       │
│ clay    ┆ other   ┆ 39443       │
│ clay    ┆ wta_125 ┆ 33175       │
│ …       ┆ …       ┆ …           │
│ null    ┆ other   ┆ 6725        │
│ clay    ┆ wta_250 ┆ 6152        │
│ grass   ┆ itf     ┆ 6033        │
│ clay    ┆ team    ┆ 5813        │
│ hard    ┆ team    ┆ 5273        │
└─────────┴─────────┴─────────────┘


alt.Chart(...)

## 12. Data Filtering Examples

Demonstrate how to filter the data for specific analyses.

In [69]:
# Example 1: Filter by recent years (2020+)
if "year" in df.columns:
    recent_matches = df.filter(pl.col("year") >= 2020)
    print(f"Matches from 2020 onwards: {recent_matches.shape[0]:,}")

# Example 2: Filter by surface (Hard court)
if "surface" in df.columns:
    hard_court = df.filter(pl.col("surface").str.to_lowercase() == "hard")
    print(f"Hard court matches: {hard_court.shape[0]:,}")

# Example 3: Filter by tournament level (Grand Slam)
if "level" in df.columns:
    grand_slams = df.filter(pl.col("level").str.contains("grand_slam", literal=False))
    print(f"Grand Slam matches: {grand_slams.shape[0]:,}")

# Example 4: Combined filter - Recent Grand Slams on Hard court
if all(col in df.columns for col in ["year", "level", "surface"]):
    filtered = df.filter(
        (pl.col("year") >= 2020) & 
        (pl.col("level").str.contains("grand_slam", literal=False)) &
        (pl.col("surface").str.to_lowercase() == "hard")
    )
    print(f"Recent Grand Slam hard court matches: {filtered.shape[0]:,}")
    
    if filtered.shape[0] > 0:
        print("\nSample of filtered data:")
        print(filtered.select(["year", "event_name", "surface", "level", "player1_name", "player2_name"]).head())

Matches from 2020 onwards: 184,998


Hard court matches: 358,835
Grand Slam matches: 41,369
Recent Grand Slam hard court matches: 3,009

Sample of filtered data:
shape: (5, 6)
┌──────┬─────────────────┬─────────┬────────────┬──────────────────┬──────────────────────────┐
│ year ┆ event_name      ┆ surface ┆ level      ┆ player1_name     ┆ player2_name             │
│ ---  ┆ ---             ┆ ---     ┆ ---        ┆ ---              ┆ ---                      │
│ i32  ┆ str             ┆ str     ┆ str        ┆ str              ┆ str                      │
╞══════╪═════════════════╪═════════╪════════════╪══════════════════╪══════════════════════════╡
│ 2020 ┆ Australian Open ┆ hard    ┆ grand_slam ┆ Ashleigh Barty   ┆ Petra Kvitova            │
│ 2020 ┆ Australian Open ┆ hard    ┆ grand_slam ┆ Garbine Muguruza ┆ Anastasia Pavlyuchenkova │
│ 2020 ┆ Australian Open ┆ hard    ┆ grand_slam ┆ Simona Halep     ┆ Anett Kontaveit          │
│ 2020 ┆ Australian Open ┆ hard    ┆ grand_slam ┆ Sofia Kenin      ┆ Ons Jabeur              

## 13. Summary and Key Insights

Summarize the key findings from the data exploration.

In [70]:
# Generate summary report
print("=" * 70)
print("WTA DATA EXPLORATION SUMMARY")
print("=" * 70)

print(f"\n📊 DATASET OVERVIEW:")
print(f"   • Total matches: {df.shape[0]:,}")
print(f"   • Total columns: {df.shape[1]}")

if "year" in df.columns:
    print(f"   • Year range: {df['year'].min()} - {df['year'].max()}")

if "player1_name" in df.columns:
    unique_players = len(set(df["player1_name"].drop_nulls().to_list() + 
                            df["player2_name"].drop_nulls().to_list()))
    print(f"   • Unique players: {unique_players:,}")

if "event_name" in df.columns:
    print(f"   • Unique tournaments: {df['event_name'].n_unique():,}")

if "surface" in df.columns:
    print(f"\n🎾 SURFACE DISTRIBUTION:")
    for row in df.group_by("surface").agg(pl.count()).sort("count", descending=True).head(5).iter_rows():
        print(f"   • {row[0]}: {row[1]:,} matches")

if "level" in df.columns:
    print(f"\n🏆 TOURNAMENT LEVELS:")
    for row in df.group_by("level").agg(pl.count()).sort("count", descending=True).head(5).iter_rows():
        print(f"   • {row[0]}: {row[1]:,} matches")

if "round" in df.columns:
    print(f"\n🔄 MOST COMMON ROUNDS:")
    for row in df.group_by("round").agg(pl.count()).sort("count", descending=True).head(5).iter_rows():
        print(f"   • {row[0]}: {row[1]:,} matches")

print("\n" + "=" * 70)
print("Exploration complete!")
print("=" * 70)

WTA DATA EXPLORATION SUMMARY

📊 DATASET OVERVIEW:
   • Total matches: 776,976
   • Total columns: 56
   • Year range: 1967 - 2026
   • Unique players: 30,365
   • Unique tournaments: 11,079

🎾 SURFACE DISTRIBUTION:
   • hard: 358,835 matches
   • clay: 328,429 matches
   • carpet: 41,462 matches
   • grass: 39,809 matches
   • None: 8,441 matches

🏆 TOURNAMENT LEVELS:
   • itf: 472,950 matches
   • other: 124,905 matches
   • wta_125: 67,955 matches
   • grand_slam: 41,369 matches
   • wta_500: 26,270 matches

🔄 MOST COMMON ROUNDS:
   • R32: 284,932 matches
   • R16: 148,555 matches
   • Q1: 85,531 matches
   • QF: 75,652 matches
   • Q2: 52,412 matches

Exploration complete!


## 14. Historical Data Analysis - Match Trends Over Time

Analyze historical trends including match volume, surface evolution, and competitive balance over the years.

In [71]:
# Historical trend: Matches per year with rolling average
if "year" in df.columns:
    matches_by_year = df.group_by("year").agg(
        pl.count().alias("match_count")
    ).sort("year")
    
    # Calculate 3-year rolling average
    matches_by_year = matches_by_year.with_columns(
        pl.col("match_count").rolling_mean(window_size=3).alias("rolling_avg_3yr")
    )
    
    # Create line chart with both actual and rolling average
    base = alt.Chart(matches_by_year.to_pandas()).encode(
        x=alt.X("year:O", title="Year", axis=alt.Axis(labelAngle=-45))
    )
    
    actual = base.mark_bar(opacity=0.6, color="steelblue").encode(
        y=alt.Y("match_count:Q", title="Number of Matches"),
        tooltip=["year", "match_count"]
    )
    
    rolling = base.mark_line(color="red", strokeWidth=3).encode(
        y="rolling_avg_3yr:Q",
        tooltip=["year", "rolling_avg_3yr"]
    )
    
    chart = (actual + rolling).properties(
        title="WTA Matches per Year (with 3-year rolling average)",
        width=700,
        height=400
    )
    
    chart.show()

alt.LayerChart(...)

In [72]:
# Historical trend: Surface evolution over decades
if "year" in df.columns and "surface" in df.columns:
    # Create decade groupings
    surface_by_decade = df.with_columns(
        ((pl.col("year") / 10).floor() * 10).cast(pl.Int32).alias("decade")
    ).group_by(["decade", "surface"]).agg(
        pl.count().alias("match_count")
    ).sort(["decade", "surface"])
    
    # Calculate percentage within each decade
    decade_totals = surface_by_decade.group_by("decade").agg(
        pl.sum("match_count").alias("decade_total")
    )
    
    surface_by_decade = surface_by_decade.join(
        decade_totals, on="decade"
    ).with_columns(
        ((pl.col("match_count") / pl.col("decade_total")) * 100).alias("percentage")
    )
    
    # Create stacked area chart
    chart = alt.Chart(surface_by_decade.to_pandas()).mark_area(opacity=0.7).encode(
        x=alt.X("decade:O", title="Decade"),
        y=alt.Y("percentage:Q", title="Percentage of Matches", stack="normalize"),
        color=alt.Color("surface:N", title="Surface"),
        tooltip=["decade", "surface", "match_count", alt.Tooltip("percentage:Q", format=".1f")]
    ).properties(
        title="Surface Type Evolution by Decade",
        width=700,
        height=400
    )
    
    chart.show()

alt.Chart(...)

## 15. Player Analysis - Performance Metrics

Deep dive into player statistics including win rates, activity levels, and performance patterns.

In [73]:
# Player win rate analysis
if "player1_name" in df.columns and "player2_name" in df.columns and "winner" in df.columns:
    # Count wins and losses for each player
    wins = df.select(
        pl.col("winner").alias("player")
    ).group_by("player").agg(
        pl.count().alias("wins")
    )
    
    # Count total matches for each player
    all_matches = pl.concat([
        df.select(pl.col("player1_name").alias("player")),
        df.select(pl.col("player2_name").alias("player"))
    ]).group_by("player").agg(
        pl.count().alias("total_matches")
    )
    
    # Combine to calculate win rates
    player_stats = all_matches.join(wins, on="player", how="left").fill_null(0)
    player_stats = player_stats.with_columns(
        ((pl.col("wins") / pl.col("total_matches")) * 100).alias("win_rate")
    ).filter(
        pl.col("total_matches") >= 20  # Filter players with at least 20 matches
    ).sort("win_rate", descending=True)
    
    print("Top 20 Players by Win Rate (min 20 matches):")
    print(player_stats.head(20))
    
    # Visualize win rate distribution
    chart = alt.Chart(player_stats.to_pandas()).mark_bar(color="forestgreen").encode(
        x=alt.X("win_rate:Q", bin=alt.Bin(maxbins=30), title="Win Rate (%)"),
        y=alt.Y("count():Q", title="Number of Players"),
        tooltip=[alt.Tooltip("win_rate:Q", bin=True), "count()"]
    ).properties(
        title="Distribution of Player Win Rates (min 20 matches)",
        width=600,
        height=400
    )
    
    chart.show()

In [74]:
# Player activity heatmap: Matches per year for top players
if "year" in df.columns and "player1_name" in df.columns:
    # Calculate player stats first (self-contained)
    all_matches = pl.concat([
        df.select(pl.col("player1_name").alias("player")),
        df.select(pl.col("player2_name").alias("player"))
    ]).group_by("player").agg(
        pl.count().alias("total_matches")
    )
    
    # Get top 20 most active players
    top_players = all_matches.sort("total_matches", descending=True).head(20)
    top_player_names = top_players["player"].to_list()
    
    # Create activity matrix
    activity_data = []
    for player in top_player_names:
        player_matches = df.filter(
            (pl.col("player1_name") == player) | (pl.col("player2_name") == player)
        )
        yearly_activity = player_matches.group_by("year").agg(pl.count().alias("matches"))
        for row in yearly_activity.iter_rows():
            activity_data.append({"player": player, "year": row[0], "matches": row[1]})
    
    activity_df = pl.DataFrame(activity_data)
    
    # Create heatmap
    chart = alt.Chart(activity_df.to_pandas()).mark_rect().encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("player:N", title="Player", sort="-x"),
        color=alt.Color("matches:Q", title="Matches", scale=alt.Scale(scheme="yelloworangered")),
        tooltip=["player", "year", "matches"]
    ).properties(
        title="Player Activity Heatmap (Top 20 Players by Match Count)",
        width=700,
        height=500
    )
    
    chart.show()

alt.Chart(...)

In [75]:
# Surface specialization analysis for top players
if "surface" in df.columns and "player1_name" in df.columns:
    # Calculate player stats first (self-contained)
    all_matches = pl.concat([
        df.select(pl.col("player1_name").alias("player")),
        df.select(pl.col("player2_name").alias("player"))
    ]).group_by("player").agg(
        pl.count().alias("total_matches")
    )
    
    # Get top 10 players
    top_10_players = all_matches.sort("total_matches", descending=True).head(10)
    
    surface_specialization = []
    for player in top_10_players["player"].to_list():
        player_matches = df.filter(
            (pl.col("player1_name") == player) | (pl.col("player2_name") == player)
        )
        surface_counts = player_matches.group_by("surface").agg(pl.count().alias("matches"))
        total = player_matches.shape[0]
        for row in surface_counts.iter_rows():
            surface_specialization.append({
                "player": player,
                "surface": row[0],
                "matches": row[1],
                "percentage": (row[1] / total) * 100
            })
    
    spec_df = pl.DataFrame(surface_specialization)
    
    # Create grouped bar chart
    chart = alt.Chart(spec_df.to_pandas()).mark_bar().encode(
        x=alt.X("percentage:Q", title="Percentage of Matches"),
        y=alt.Y("player:N", title="Player", sort="-x"),
        color=alt.Color("surface:N", title="Surface"),
        tooltip=["player", "surface", "matches", alt.Tooltip("percentage:Q", format=".1f")]
    ).properties(
        title="Surface Specialization - Top 10 Players",
        width=700,
        height=400
    )
    
    chart.show()

alt.Chart(...)

## 16. Tournament Analysis

Analyze tournament patterns, competitive balance, and geographic distribution.

In [76]:
# Top tournaments by match count
if "event_name" in df.columns:
    # Use match_date for year calculations since 'year' column may have nulls
    tournament_stats = df.filter(pl.col("match_date").is_not_null()).group_by("event_name").agg(
        pl.count().alias("total_matches"),
        pl.col("match_date").dt.year().n_unique().alias("years_held"),
        pl.col("match_date").dt.year().min().alias("first_year"),
        pl.col("match_date").dt.year().max().alias("last_year")
    ).sort("total_matches", descending=True)
    
    print("Top 15 Tournaments by Match Count:")
    print(tournament_stats.head(15))
    
    # Visualize top 15
    top_15_tournaments = tournament_stats.head(15)
    
    chart = alt.Chart(top_15_tournaments.to_pandas()).mark_bar(color="mediumpurple").encode(
        x=alt.X("total_matches:Q", title="Total Matches"),
        y=alt.Y("event_name:N", title="Tournament", sort="-x"),
        tooltip=["event_name", "total_matches", "years_held", "first_year", "last_year"]
    ).properties(
        title="Top 15 Tournaments by Total Matches",
        width=600,
        height=500
    )
    
    chart.show()

Top 15 Tournaments by Match Count:
shape: (15, 5)
┌─────────────────────┬───────────────┬────────────┬────────────┬───────────┐
│ event_name          ┆ total_matches ┆ years_held ┆ first_year ┆ last_year │
│ ---                 ┆ ---           ┆ ---        ┆ ---        ┆ ---       │
│ str                 ┆ u32           ┆ u32        ┆ i32        ┆ i32       │
╞═════════════════════╪═══════════════╪════════════╪════════════╪═══════════╡
│ W15 Monastir        ┆ 14224         ┆ 8          ┆ 2019       ┆ 2026      │
│ W15 Antalya         ┆ 11507         ┆ 8          ┆ 2019       ┆ 2026      │
│ Wimbledon           ┆ 10705         ┆ 57         ┆ 1968       ┆ 2025      │
│ Roland Garros       ┆ 10468         ┆ 59         ┆ 1968       ┆ 2026      │
│ US Open             ┆ 9359          ┆ 51         ┆ 1968       ┆ 2018      │
│ …                   ┆ …             ┆ …          ┆ …          ┆ …         │
│ Miami               ┆ 3546          ┆ 31         ┆ 1968       ┆ 2026      │
│ W15 Heraklio

alt.Chart(...)

In [77]:
# Tournament level vs surface analysis
if "level" in df.columns and "surface" in df.columns:
    level_surface = df.group_by(["level", "surface"]).agg(
        pl.count().alias("match_count")
    )
    
    # Create bubble chart
    chart = alt.Chart(level_surface.to_pandas()).mark_circle(opacity=0.7).encode(
        x=alt.X("level:N", title="Tournament Level"),
        y=alt.Y("surface:N", title="Surface"),
        size=alt.Size("match_count:Q", title="Matches", scale=alt.Scale(range=[50, 1000])),
        color=alt.Color("match_count:Q", scale=alt.Scale(scheme="viridis")),
        tooltip=["level", "surface", "match_count"]
    ).properties(
        title="Tournament Level vs Surface Distribution",
        width=600,
        height=400
    )
    
    chart.show()

alt.Chart(...)

## 17. Future Match Data Summary

Analyze upcoming fixtures and scheduled matches.

In [78]:
# Analyze future fixtures
if "record_type" in df.columns:
    # Separate completed matches from fixtures
    fixtures = df.filter(pl.col("record_type") == "fixture")
    completed = df.filter(pl.col("record_type") == "completed")
    
    print(f"Dataset Summary:")
    print(f"  Completed matches: {completed.shape[0]:,}")
    print(f"  Future fixtures: {fixtures.shape[0]:,}")
    print(f"  Total records: {df.shape[0]:,}")
    
    if fixtures.shape[0] > 0:
        print("\n" + "="*50)
        print("FUTURE FIXTURES ANALYSIS")
        print("="*50)
        
        # Fixtures by tournament
        if "event_name" in fixtures.columns:
            fixture_by_tournament = fixtures.group_by("event_name").agg(
                pl.count().alias("fixture_count")
            ).sort("fixture_count", descending=True)
            
            print("\nTop 10 Tournaments with Upcoming Fixtures:")
            print(fixture_by_tournament.head(10))
        
        # Fixtures by surface
        if "surface" in fixtures.columns:
            fixture_by_surface = fixtures.group_by("surface").agg(
                pl.count().alias("fixture_count")
            ).sort("fixture_count", descending=True)
            
            print("\nUpcoming Fixtures by Surface:")
            print(fixture_by_surface)
        
        # Fixtures by level
        if "level" in fixtures.columns:
            fixture_by_level = fixtures.group_by("level").agg(
                pl.count().alias("fixture_count")
            ).sort("fixture_count", descending=True)
            
            print("\nUpcoming Fixtures by Tournament Level:")
            print(fixture_by_level)
else:
    print("Note: 'record_type' column not found. Dataset may contain only completed matches.")
    print(f"Total matches in dataset: {df.shape[0]:,}")

Dataset Summary:
  Completed matches: 776,962
  Future fixtures: 14
  Total records: 776,976

FUTURE FIXTURES ANALYSIS

Top 10 Tournaments with Upcoming Fixtures:
shape: (2, 2)
┌─────────────┬───────────────┐
│ event_name  ┆ fixture_count │
│ ---         ┆ ---           │
│ str         ┆ u32           │
╞═════════════╪═══════════════╡
│ Iași Open   ┆ 7             │
│ Athens Open ┆ 7             │
└─────────────┴───────────────┘

Upcoming Fixtures by Surface:
shape: (1, 2)
┌─────────┬───────────────┐
│ surface ┆ fixture_count │
│ ---     ┆ ---           │
│ str     ┆ u32           │
╞═════════╪═══════════════╡
│ null    ┆ 14            │
└─────────┴───────────────┘

Upcoming Fixtures by Tournament Level:
shape: (1, 2)
┌───────┬───────────────┐
│ level ┆ fixture_count │
│ ---   ┆ ---           │
│ str   ┆ u32           │
╞═══════╪═══════════════╡
│ other ┆ 14            │
└───────┴───────────────┘


In [93]:
fixtures

record_type,record_id,is_fixture,fixture_id,scheduled_on,scheduled_at,schedule_date_source,fixture_observed_on,fixture_source,fixture_source_match_id,match_id,event_id,tour,year,discipline,event_name,level,level_detail,source_level,surface,indoor,event_start_date,event_city,event_country,draw,round,round_order,bracket_slot,played_on,played_on_precision,player1_id,player1_name,player1_country,player2_id,player2_name,player2_country,winner_id,loser_id,winner_side,player1_seed,player2_seed,player1_entry,player2_entry,player1_rank,player2_rank,player1_rank_points,player2_rank_points,status,termination,score,best_of,first_observed_on,last_updated_on,preferred_source,source_count,match_date
str,str,bool,str,date,datetime[μs],str,date,str,str,str,str,str,i32,str,str,str,str,str,str,bool,date,str,str,str,str,i16,str,date,str,str,str,str,str,str,str,str,str,i8,str,str,str,str,i32,i32,i32,i32,str,str,str,i8,date,date,str,i16,date
"""fixture""","""fixture-4632b3085dad2650d464d4…",true,"""fixture-4632b3085dad2650d464d4…",null,null,"""draw_slot""",2026-07-16,"""wikimedia""","""83670759:3:2:2""",null,"""event:wta:4843a652157e541131c5""","""wta""",null,"""singles""","""Iași Open""","""other""",null,null,null,null,null,null,null,"""main""","""R64""",null,null,null,null,"""wta:260581""","""Alevtina Ibragimova""",null,"""wta:211651""","""Paula Badosa""",null,null,null,null,null,null,null,null,null,null,null,null,"""tentative""",null,null,null,null,null,null,null,null
"""fixture""","""fixture-63aa6c70cb14096d3b3f99…",true,"""fixture-63aa6c70cb14096d3b3f99…",null,null,"""draw_slot""",2026-07-16,"""wikimedia""","""83670759:3:2:1""",null,"""event:wta:4843a652157e541131c5""","""wta""",null,"""singles""","""Iași Open""","""other""",null,null,null,null,null,null,null,"""main""","""R64""",null,null,null,null,"""wta:215910""","""Panna Udvardy""",null,"""wta:203514""","""Katarzyna Kawa""",null,null,null,null,null,null,null,null,null,null,null,null,"""tentative""",null,null,null,null,null,null,null,null
"""fixture""","""fixture-7f5d41cb1a6b96152ada58…",true,"""fixture-7f5d41cb1a6b96152ada58…",null,null,"""draw_slot""",2026-07-16,"""wikimedia""","""83670759:2:3:1""",null,"""event:wta:4843a652157e541131c5""","""wta""",null,"""singles""","""Iași Open""","""other""",null,null,null,null,null,null,null,"""main""","""QF""",null,null,null,null,"""wta:201709""","""Yulia Putintseva""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""tentative""",null,null,null,null,null,null,null,null
"""fixture""","""fixture-9ac94c12c19c6b5cdceff2…",true,"""fixture-9ac94c12c19c6b5cdceff2…",null,null,"""draw_slot""",2026-07-16,"""wikimedia""","""83670759:2:2:3""",null,"""event:wta:4843a652157e541131c5""","""wta""",null,"""singles""","""Iași Open""","""other""",null,null,null,null,null,null,null,"""main""","""R64""",null,null,null,null,"""wta:220716""","""Oleksandra Oliynykova""",null,"""wta:239389""","""Elena Pridankina""",null,null,null,null,null,null,null,null,null,null,null,null,"""tentative""",null,null,null,null,null,null,null,null
"""fixture""","""fixture-b7d7091354e47e689f5625…",true,"""fixture-b7d7091354e47e689f5625…",null,null,"""draw_slot""",2026-07-16,"""wikimedia""","""83670759:2:3:2""",null,"""event:wta:4843a652157e541131c5""","""wta""",null,"""singles""","""Iași Open""","""other""",null,null,null,null,null,null,null,"""main""","""QF""",null,null,null,null,null,null,null,"""wta:216262""","""Clara Burel""",null,null,null,null,null,null,null,null,null,null,null,null,"""tentative""",null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""fixture""","""fixture-3a4a86a4631e3197153079…",true,"""fixture-3a4a86a4631e3197153079…",null,null,"""draw_slot""",2026-07-16,"""wikimedia""","""83663463:3:3:1""",null,"""event:wta:83ba7f2d813f738b4951""","""wta""",null,"""singles""","""Athens Open""","""other""",null,null,null,null,null,null,null,"""main""","""QF

In [89]:
fixtures.head(5)

record_type,record_id,is_fixture,fixture_id,scheduled_on,scheduled_at,schedule_date_source,fixture_observed_on,fixture_source,fixture_source_match_id,match_id,event_id,tour,year,discipline,event_name,level,level_detail,source_level,surface,indoor,event_start_date,event_city,event_country,draw,round,round_order,bracket_slot,played_on,played_on_precision,player1_id,player1_name,player1_country,player2_id,player2_name,player2_country,winner_id,loser_id,winner_side,player1_seed,player2_seed,player1_entry,player2_entry,player1_rank,player2_rank,player1_rank_points,player2_rank_points,status,termination,score,best_of,first_observed_on,last_updated_on,preferred_source,source_count,match_date
str,str,bool,str,date,datetime[μs],str,date,str,str,str,str,str,i32,str,str,str,str,str,str,bool,date,str,str,str,str,i16,str,date,str,str,str,str,str,str,str,str,str,i8,str,str,str,str,i32,i32,i32,i32,str,str,str,i8,date,date,str,i16,date
"""fixture""","""fixture-4632b3085dad2650d464d4…",true,"""fixture-4632b3085dad2650d464d4…",null,null,"""draw_slot""",2026-07-16,"""wikimedia""","""83670759:3:2:2""",null,"""event:wta:4843a652157e541131c5""","""wta""",null,"""singles""","""Iași Open""","""other""",null,null,null,null,null,null,null,"""main""","""R64""",null,null,null,null,"""wta:260581""","""Alevtina Ibragimova""",null,"""wta:211651""","""Paula Badosa""",null,null,null,null,null,null,null,null,null,null,null,null,"""tentative""",null,null,null,null,null,null,null,null
"""fixture""","""fixture-63aa6c70cb14096d3b3f99…",true,"""fixture-63aa6c70cb14096d3b3f99…",null,null,"""draw_slot""",2026-07-16,"""wikimedia""","""83670759:3:2:1""",null,"""event:wta:4843a652157e541131c5""","""wta""",null,"""singles""","""Iași Open""","""other""",null,null,null,null,null,null,null,"""main""","""R64""",null,null,null,null,"""wta:215910""","""Panna Udvardy""",null,"""wta:203514""","""Katarzyna Kawa""",null,null,null,null,null,null,null,null,null,null,null,null,"""tentative""",null,null,null,null,null,null,null,null
"""fixture""","""fixture-7f5d41cb1a6b96152ada58…",true,"""fixture-7f5d41cb1a6b96152ada58…",null,null,"""draw_slot""",2026-07-16,"""wikimedia""","""83670759:2:3:1""",null,"""event:wta:4843a652157e541131c5""","""wta""",null,"""singles""","""Iași Open""","""other""",null,null,null,null,null,null,null,"""main""","""QF""",null,null,null,null,"""wta:201709""","""Yulia Putintseva""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""tentative""",null,null,null,null,null,null,null,null
"""fixture""","""fixture-9ac94c12c19c6b5cdceff2…",true,"""fixture-9ac94c12c19c6b5cdceff2…",null,null,"""draw_slot""",2026-07-16,"""wikimedia""","""83670759:2:2:3""",null,"""event:wta:4843a652157e541131c5""","""wta""",null,"""singles""","""Iași Open""","""other""",null,null,null,null,null,null,null,"""main""","""R64""",null,null,null,null,"""wta:220716""","""Oleksandra Oliynykova""",null,"""wta:239389""","""Elena Pridankina""",null,null,null,null,null,null,null,null,null,null,null,null,"""tentative""",null,null,null,null,null,null,null,null
"""fixture""","""fixture-b7d7091354e47e689f5625…",true,"""fixture-b7d7091354e47e689f5625…",null,null,"""draw_slot""",2026-07-16,"""wikimedia""","""83670759:2:3:2""",null,"""event:wta:4843a652157e541131c5""","""wta""",null,"""singles""","""Iași Open""","""other""",null,null,null,null,null,null,null,"""main""","""QF""",null,null,null,null,null,null,null,"""wta:216262""","""Clara Burel""",null,null,null,null,null,null,null,null,null,null,null,null,"""tentative""",null,null,null,null,null,null,null,null


In [95]:
completed.tail(5)

record_type,record_id,is_fixture,fixture_id,scheduled_on,scheduled_at,schedule_date_source,fixture_observed_on,fixture_source,fixture_source_match_id,match_id,event_id,tour,year,discipline,event_name,level,level_detail,source_level,surface,indoor,event_start_date,event_city,event_country,draw,round,round_order,bracket_slot,played_on,played_on_precision,player1_id,player1_name,player1_country,player2_id,player2_name,player2_country,winner_id,loser_id,winner_side,player1_seed,player2_seed,player1_entry,player2_entry,player1_rank,player2_rank,player1_rank_points,player2_rank_points,status,termination,score,best_of,first_observed_on,last_updated_on,preferred_source,source_count,match_date
str,str,bool,str,date,datetime[μs],str,date,str,str,str,str,str,i32,str,str,str,str,str,str,bool,date,str,str,str,str,i16,str,date,str,str,str,str,str,str,str,str,str,i8,str,str,str,str,i32,i32,i32,i32,str,str,str,i8,date,date,str,i16,date
"""completed""","""match:wta:f42807db691bf7d471c0""",false,null,null,null,null,null,null,null,"""match:wta:f42807db691bf7d471c0""","""event:wta:fcc558086d3354417282""","""wta""",null,"""singles""","""Wimbledon Championships""","""other""",null,null,null,false,null,null,null,"""main""","""R64""",999,"""b3-r2-m1""",null,"""unknown""","""wta:214096""","""Karolina Muchova""","""CZE""","""wikimedia:Zhang_Shuai""","""Zhang Shuai""","""CHN""","""wta:214096""","""wikimedia:Zhang_Shuai""",1,null,null,null,null,null,null,null,null,"""completed""",null,"""6-3 6-2""",null,2026-07-16,2026-07-16,"""wikimedia""",1,null
"""completed""","""match:wta:f4e15e18d50d9961b00a""",false,null,null,null,null,null,null,null,"""match:wta:f4e15e18d50d9961b00a""","""event:wta:fcc558086d3354417282""","""wta""",null,"""singles""","""Wimbledon Championships""","""other""",null,null,null,false,null,null,null,"""main""","""R128""",999,"""b2-r1-m4""",null,"""unknown""","""wta:222045""","""Antonia Ruzic""","""CRO""","""wta:221985""","""Darja Semenistaja""","""LAT""","""wta:222045""","""wta:221985""",1,null,null,null,null,null,null,null,null,"""completed""",null,"""6-3 3-6 6-3""",null,2026-07-16,2026-07-16,"""wikimedia""",1,null
"""completed""","""match:wta:f89db264996f99d40adb""",false,null,null,null,null,null,null,null,"""match:wta:f89db264996f99d40adb""","""event:wta:fcc558086d3354417282""","""wta""",null,"""singles""","""Wimbledon Championships""","""other""",null,null,null,false,null,null,null,"""main""","""R128""",999,"""b6-r1-m3""",null,"""unknown""","""wta:201548""","""Ajla Tomljanovic""","""AUS""","""wta:213734""","""Mariam Bolkvadze""","""GEO""","""wta:213734""","""wta:201548""",2,null,null,null,null,null,null,null,null,"""completed""",null,"""6-2 2-6 6-4""",null,2026-07-16,2026-07-16,"""wikimedia""",1,null
"""completed""","""match:wta:f9633b60cd962fb564ec""",false,null,null,null,null,null,null,null,"""match:wta:f9633b60cd962fb564ec""","""event:wta:fcc558086d3354417282""","""wta""",null,"""singles""","""Wimbledon Championships""","""other""",null,null,null,false,null,null,null,"""main""","""R32""",999,"""b9-r3-m2""",null,"""unknown""","""wta:210722""","""Elise Mertens""","""BEL""","""wta:214981""","""Elena Rybakina""","""KAZ""","""wta:210722""","""wta:214981""",1,null,null,null,null,null,null,null,null,"""completed""",null,"""7-6 6-1""",null,2026-07-16,2026-07-16,"""wikimedia""",1,null
"""completed""","""match:wta:f97320be02c416fb1bc2""",false,null,null,null,null,null,null,null,"""match:wta:f97320be02c416fb1bc2""","""event:wta:fcc558086d3354417282""","""wta""",null,"""singles""","""Wimbledon Championships""","""other""",null,null,null,false,null,null,null,"""main""","""R128""",999,"""b2-r1-m7""",null,"""unknown""","""wta:214181""","""Anastasia Gasanova""","""RUS""","""wta:215306""","""Emiliana Arango""","""COL""","""wta:214181""","""wta:215306""",1,null,null,null,null,null,null,null,null,"""completed""",null,"""6-3 6-1""",null,2026-07-16,2026-07-16,"""wikimedia""",1,null


In [79]:
# Visualize future fixtures if available
if "record_type" in df.columns:
    fixtures = df.filter(pl.col("record_type") == "fixture")
    
    if fixtures.shape[0] > 0 and "surface" in fixtures.columns:
        fixture_surface = fixtures.group_by("surface").agg(
            pl.count().alias("fixture_count")
        )
        
        # Create donut chart for fixtures by surface
        chart = alt.Chart(fixture_surface.to_pandas()).mark_arc(innerRadius=80).encode(
            theta=alt.Theta("fixture_count:Q", stack=True),
            color=alt.Color("surface:N", title="Surface"),
            tooltip=["surface", "fixture_count"]
        ).properties(
            title="Upcoming Fixtures by Surface",
            width=400,
            height=400
        )
        
        chart.show()
    else:
        print("No future fixtures found in the dataset.")

alt.Chart(...)

In [80]:
# Compare completed vs fixtures over time (if scheduled dates available)
if "record_type" in df.columns and "scheduled_on" in df.columns:
    fixtures = df.filter(pl.col("record_type") == "fixture")
    
    if fixtures.shape[0] > 0:
        # Extract month from scheduled dates
        fixtures_by_month = fixtures.with_columns(
            pl.col("scheduled_on").dt.month().alias("month")
        ).group_by("month").agg(
            pl.count().alias("fixture_count")
        ).sort("month")
        
        # Month names
        month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", 
                       "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
        fixtures_by_month = fixtures_by_month.with_columns(
            pl.col("month").map_elements(lambda x: month_names[x-1], return_dtype=pl.String).alias("month_name")
        )
        
        chart = alt.Chart(fixtures_by_month.to_pandas()).mark_line(point=True, color="darkorange").encode(
            x=alt.X("month_name:N", title="Month", sort=month_names),
            y=alt.Y("fixture_count:Q", title="Number of Fixtures"),
            tooltip=["month_name", "fixture_count"]
        ).properties(
            title="Upcoming Fixtures by Month",
            width=700,
            height=400
        )
        
        chart.show()
else:
    print("Scheduled date information not available for future fixtures.")

alt.Chart(...)

## 18. Comprehensive Summary Dashboard

A final summary combining all key insights from the analysis.

In [81]:
# Generate comprehensive summary report
print("=" * 80)
print("WTA TENNIS DATA - COMPREHENSIVE ANALYSIS SUMMARY")
print("=" * 80)

print("\n📊 DATASET OVERVIEW:")
print(f"   • Total records: {df.shape[0]:,}")
print(f"   • Total columns: {df.shape[1]}")

if "record_type" in df.columns:
    completed_count = df.filter(pl.col("record_type") == "completed").shape[0]
    fixture_count = df.filter(pl.col("record_type") == "fixture").shape[0]
    print(f"   • Completed matches: {completed_count:,}")
    print(f"   • Future fixtures: {fixture_count:,}")

if "year" in df.columns:
    year_min = df['year'].drop_nulls().min()
    year_max = df['year'].drop_nulls().max()
    if year_min is not None and year_max is not None:
        print(f"   • Year range: {year_min} - {year_max}")

if "player1_name" in df.columns:
    unique_players = len(set(df["player1_name"].drop_nulls().to_list() + 
                            df["player2_name"].drop_nulls().to_list()))
    print(f"   • Unique players: {unique_players:,}")

if "event_name" in df.columns:
    print(f"   • Unique tournaments: {df['event_name'].n_unique():,}")

print("\n🎾 SURFACE DISTRIBUTION:")
if "surface" in df.columns:
    for row in df.group_by("surface").agg(pl.count()).sort("count", descending=True).iter_rows():
        pct = (row[1] / df.shape[0]) * 100
        print(f"   • {row[0]}: {row[1]:,} matches ({pct:.1f}%)")

print("\n🏆 TOURNAMENT LEVELS:")
if "level" in df.columns:
    for row in df.group_by("level").agg(pl.count()).sort("count", descending=True).head(5).iter_rows():
        pct = (row[1] / df.shape[0]) * 100
        print(f"   • {row[0]}: {row[1]:,} matches ({pct:.1f}%)")

print("\n👥 TOP PLAYERS (by match count):")
if "player1_name" in df.columns:
    # Calculate top players on the fly
    all_matches = pl.concat([
        df.select(pl.col("player1_name").alias("player")),
        df.select(pl.col("player2_name").alias("player"))
    ]).group_by("player").agg(pl.count().alias("total_matches")).sort("total_matches", descending=True)
    
    top_5 = all_matches.head(5)
    for row in top_5.iter_rows():
        print(f"   • {row[0]}: {row[1]:,} matches")

print("\n📅 HISTORICAL TRENDS:")
if "year" in df.columns:
    yearly_counts = df.filter(pl.col("year").is_not_null()).group_by("year").agg(pl.count()).sort("year")
    if yearly_counts.shape[0] > 0:
        earliest = yearly_counts.head(1).row(0)
        latest = yearly_counts.tail(1).row(0)
        print(f"   • Earliest year ({earliest[0]}): {earliest[1]:,} matches")
        print(f"   • Latest year ({latest[0]}): {latest[1]:,} matches")
        
        # Calculate growth
        if earliest[1] > 0:
            growth = ((latest[1] - earliest[1]) / earliest[1]) * 100
            print(f"   • Growth: {growth:+.1f}%")

print("\n🔮 FUTURE FIXTURES:")
if "record_type" in df.columns:
    fixtures = df.filter(pl.col("record_type") == "fixture")
    if fixtures.shape[0] > 0:
        print(f"   • Total upcoming fixtures: {fixtures.shape[0]:,}")
        if "surface" in fixtures.columns:
            top_surface = fixtures.group_by("surface").agg(pl.count()).sort("count", descending=True).row(0)
            print(f"   • Most common surface: {top_surface[0]} ({top_surface[1]:,} fixtures)")
    else:
        print("   • No future fixtures in current dataset")

print("\n" + "=" * 80)
print("Analysis complete! Use the filtering examples to explore specific subsets.")
print("=" * 80)

WTA TENNIS DATA - COMPREHENSIVE ANALYSIS SUMMARY

📊 DATASET OVERVIEW:
   • Total records: 776,976
   • Total columns: 56
   • Completed matches: 776,962
   • Future fixtures: 14
   • Year range: 1967 - 2026
   • Unique players: 30,365
   • Unique tournaments: 11,079

🎾 SURFACE DISTRIBUTION:
   • hard: 358,835 matches (46.2%)
   • clay: 328,429 matches (42.3%)
   • carpet: 41,462 matches (5.3%)
   • grass: 39,809 matches (5.1%)
   • None: 8,441 matches (1.1%)

🏆 TOURNAMENT LEVELS:
   • itf: 472,950 matches (60.9%)
   • other: 124,905 matches (16.1%)
   • wta_125: 67,955 matches (8.7%)
   • grand_slam: 41,369 matches (5.3%)
   • wta_500: 26,270 matches (3.4%)

👥 TOP PLAYERS (by match count):
   • Martina Navratilova: 1,735 matches
   • Chris Evert: 1,503 matches
   • Virginia Wade: 1,294 matches
   • Tatjana Maria: 1,264 matches
   • Varvara Lepchenko: 1,226 matches

📅 HISTORICAL TRENDS:
   • Earliest year (1967): 21 matches
   • Latest year (2026): 10,068 matches
   • Growth: +47842.9%
